# 08 - Revue bancaire experte: IFRS 9, Bale et stress testing

            Ce notebook complete les notebooks 01 a 07 avec une vue de plateforme bancaire:
            date d'arrete, segmentation reglementaire, RWA, capital stack, stress testing a 1 an,
            projection T1-T5, gouvernance modele et audit trail.

            Objectif de soutenance: montrer que le projet ne calcule pas seulement une ECL totale,
            mais simule une decision bancaire: perte attendue l'annee prochaine, resistance capital
            et actions de management.

## Resultats generes automatiquement

Cette synthese est inseree par `scripts/generate_banking_expert_review.py` pour que
les resultats soient visibles dans le notebook meme avant re-execution.

### Capital stack Bale

```text
  EAD_M$   RWA_M$  RWA_density_%  CET1_ratio_%  Total_capital_ratio_%  Leverage_ratio_%
34004.21 29063.66          85.47          12.0                   15.5             10.26
```

### Matrice stress testing a 1 an - scenarios les plus critiques

```text
              Scenario  Expected_loss_Y1_M$  CET1_post_stress_%  Capital_shortfall_7pct_M$                                                                                   Management_action
        Crise COVID-19              9267.44              -9.661                    5172.85 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
            Crise 2008              8125.72              -7.475                    4261.05 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
                Severe              5619.74              -1.555                    2455.35 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
             Choc Taux              4958.18               0.060                    1956.66 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
               Adverse              4700.91               0.699                    1774.96 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
Inflation Ukraine 2022              2665.53               5.939                     289.72               Limiter dividendes, reduire croissance RWA, renforcer recouvrement et pricing risque.
      Choc Marche 2018               934.29              10.573                       0.00                                Maintenir coussins, suivi normal et revue trimestrielle des limites.
```

### Projection dynamique T1-T5 - crise 2008

```text
 year   ECL_M$   RWA_M$  CET1_%  PD_mean_%  UNRATE  GDPC1_growth
    1 11928.76 29437.31  -7.475     90.112  10.000       -4.3000
    2 13334.69 30336.66 -10.497     87.736   9.175       -3.4337
    3 13843.52 30948.96 -11.440     84.622   8.350       -2.5675
    4 13540.04 30946.31 -11.441     80.628   7.525       -1.7012
    5 12191.08 30383.87 -11.653     75.047   6.700       -0.8350
```

In [ ]:
import sys
            from pathlib import Path

            import pandas as pd

            ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
            sys.path.insert(0, str(ROOT))

            from src.banking_core import (
                capital_actions_from_shortfall,
                compute_basel_metrics,
                enrich_banking_view,
                ensure_model_registry,
                load_audit_log,
                log_audit_event,
                segment_summary,
            )
            from src.scenario_projection import compute_dynamic_projection
            from src.stress_scenarios import ACTIVE_STRESS_SCENARIOS, SCENARIO_LABELS

            OUTPUT_DIR = ROOT / "outputs"
            RESULTS_DIR = OUTPUT_DIR / "results"
            RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Portefeuille a date d'arrete

            Une banque ne travaille pas sur tout l'historique brut. Elle fige un portefeuille a une
            date de reporting, puis calcule ECL, RWA et stress testing sur cette photographie.

In [ ]:
df_full = pd.read_parquet(OUTPUT_DIR / "loan_clean.parquet")
            ecl = pd.read_parquet(OUTPUT_DIR / "ecl_baseline.parquet")
            cols = [c for c in ["issue_d", "purpose", "grade", "stage", "loan_amnt", "EAD", "LGD", "term_months"] if c in df_full.columns]
            df = df_full[cols].copy()
            df["PD_pred"] = ecl["PD_pred"].values
            df["ECL_ifrs9"] = ecl["ECL_ifrs9"].values
            del df_full, ecl

            reporting_date = pd.to_datetime(df["issue_d"]).max() if "issue_d" in df.columns else pd.Timestamp.today()
            banking_df, reporting_date = enrich_banking_view(df, reporting_date)

            print(f"Date d'arrete: {reporting_date.date()}")
            print(f"Nombre de prets: {len(banking_df):,}")
            print(f"EAD total: {banking_df['EAD'].sum()/1e6:,.1f} M$")
            print(f"ECL baseline: {banking_df['ECL_ifrs9'].sum()/1e6:,.1f} M$")
            banking_df.head()

## 2. Segmentation reglementaire et RWA

            La segmentation permet de sortir du simple portefeuille agrege. Elle porte la logique
            Bale: chaque segment a un risk weight et contribue differemment au RWA.

In [ ]:
seg = segment_summary(banking_df)
            seg.to_csv(RESULTS_DIR / "regulatory_segment_summary.csv", index=False)
            seg

In [ ]:
basel_metrics = compute_basel_metrics(banking_df, cet1_ratio=0.12)
            basel_df = pd.DataFrame([basel_metrics])
            basel_df.to_csv(RESULTS_DIR / "basel_capital_metrics.csv", index=False)
            basel_df.T

## 3. Stress testing a 1 an: perte, CET1 et decision

            Le coeur bancaire du stress testing est la question suivante:
            si un choc arrive l'annee prochaine, combien la banque doit provisionner,
            est-ce que le CET1 resiste, et quelle action doit etre declenchee?

In [ ]:
rows = []
            for scenario_key, scenario in ACTIVE_STRESS_SCENARIOS.items():
                projection, stage_projection = compute_dynamic_projection(
                    banking_df,
                    scenario,
                    horizon_years=1,
                    profile="front_loaded",
                    cet1_initial=0.12,
                    tax_rate=0.30,
                    scenario_name=scenario_key,
                )
                row = projection.iloc[0].to_dict()
                shortfall = max((7.0 - row["CET1_%"]) / 100 * row["RWA_M$"], 0)
                rows.append({
                    "scenario_key": scenario_key,
                    "Scenario": SCENARIO_LABELS.get(scenario_key, scenario_key),
                    "Expected_loss_Y1_M$": row["Annual_provision_M$"],
                    "ECL_Y1_M$": row["ECL_M$"],
                    "RWA_Y1_M$": row["RWA_M$"],
                    "CET1_post_stress_%": row["CET1_%"],
                    "PD_mean_%": row["PD_mean_%"],
                    "LGD_mean_%": row["LGD_mean_%"],
                    "Stage3_%": stage_projection.iloc[0]["Stage 3_%"],
                    "Capital_shortfall_7pct_M$": round(shortfall, 2),
                    "Management_action": capital_actions_from_shortfall(row["CET1_%"], shortfall),
                })

            stress_matrix = pd.DataFrame(rows).sort_values("CET1_post_stress_%")
            stress_matrix.to_csv(RESULTS_DIR / "one_year_stress_decision_matrix.csv", index=False)
            stress_matrix

## 4. Projection dynamique T1-T5

            Cette section montre la trajectoire pluriannuelle d'un scenario severe: macro,
            ECL, RWA, CET1 et migration de stages. Elle repond a la critique du calcul
            uniquement total et non temporel.

In [ ]:
projection, stage_projection = compute_dynamic_projection(
                banking_df,
                ACTIVE_STRESS_SCENARIOS["crisis_2008"],
                horizon_years=5,
                profile="front_loaded",
                cet1_initial=0.12,
                tax_rate=0.30,
                scenario_name="crisis_2008_expert",
            )
            projection.to_csv(RESULTS_DIR / "dynamic_projection_expert.csv", index=False)
            stage_projection.to_csv(RESULTS_DIR / "dynamic_stage_projection_expert.csv", index=False)
            projection

In [ ]:
stage_projection

## 5. Gouvernance: model registry et audit trail

            Une plateforme bancaire doit tracer la version des modeles, le statut de validation,
            le scenario execute et les sorties principales.

In [ ]:
registry = ensure_model_registry()
            registry

In [ ]:
worst = stress_matrix.iloc[0]
            run_id = log_audit_event(
                event_type="notebook_08_banking_expert_review",
                user="notebook_user",
                parameters={
                    "reporting_date": str(reporting_date.date()),
                    "n_scenarios": len(stress_matrix),
                    "capital_framework": "Basel standardized simplified",
                },
                outputs={
                    "worst_scenario": worst["Scenario"],
                    "expected_loss_y1_m": worst["Expected_loss_Y1_M$"],
                    "cet1_post_stress_pct": worst["CET1_post_stress_%"],
                    "management_action": worst["Management_action"],
                },
            )
            print(f"Run audite: {run_id}")
            load_audit_log(limit=10)

## Conclusion bancaire

            Le projet couvre maintenant une chaine complete:

            macro scenario -> PD/LGD/EAD -> ECL -> RWA -> CET1 -> decision management -> audit trail.

            Les limites a expliquer restent: risk weights simplifiees, donnees Lending Club non bancaires
            au sens prudentiel complet, absence de vraie base comptable et calibration encore academique.